# 08 — DataOps & Production Readiness

This notebook assesses the operational readiness of the Medalloan pipeline based on repository evidence. It focuses on layered testing, CI/CD, configuration, deployment checklists, rollback, and peer review.

**Enterprise control:** no untracked manual change is allowed in production; artifacts, configuration, approvals, and validation must be reproducible.

## Learning content

- **Git branching and pull-request review:** changes must go through a branch and review.
- **Testing:** unit, integration, data regression, and pipeline regression tests.
- **CI/CD:** CI validates changes; CD produces promotable artifacts.
- **Promotion and rollback:** configuration is separated from code; a release can be restored to a previous artifact.
- **Cost/performance:** measure throughput, duration, chunk size, concurrency, and operating cost.
- **Definition of Done:** the pipeline is complete only when quality, observability, security, testing, approval, and recovery are validated.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import time

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)
assert (PROJECT_ROOT / "src" / "medalloan").exists()
assert (PROJECT_ROOT / "requirements.txt").exists()

Project root: c:\Users\hadarmawan\Documents\medalloan


## Repository evidence inventory

The first step is to record what actually exists. A planned control is not counted as implemented until there is repository evidence or a passing execution result. This notebook uses a strict standard: documentation alone does not count as completion.

In [ ]:
evidence = {
    "unit_tests": "DONE" if (PROJECT_ROOT / "tests" / "test_unit_config.py").exists() else "MISSING",
    "data_regression_tests": "DONE" if (PROJECT_ROOT / "tests" / "test_data_regression.py").exists() else "MISSING",
    "postgres_integration_test": "DONE" if (PROJECT_ROOT / "tests" / "test_integration_postgres.py").exists() else "MISSING",
    "ci_workflow": "DONE" if (PROJECT_ROOT / ".github" / "workflows" / "ci.yml").exists() else "MISSING",
    "cd_artifact_workflow": "PARTIAL" if (PROJECT_ROOT / ".github" / "workflows" / "cd.yml").exists() else "MISSING",
    "git_branching_and_pr_review": "MISSING",
    "full_pipeline_regression": "PARTIAL",
    "environment_promotion": "MISSING",
    "rollback_automation": "MISSING",
    "peer_review_record": "MISSING",
}
for name, present in evidence.items():
    print(evidence[name], name)
assert all(evidence[name] == "DONE" for name in ["unit_tests", "data_regression_tests", "postgres_integration_test", "ci_workflow"])

## Testing pyramid

Unit tests provide fast feedback. Data regression tests protect the dataset contract. Integration tests validate PostgreSQL connectivity and behavior. CI combines all of these checks for every change.

In [2]:
test_command = [sys.executable, "-m", "pytest", "tests", "-p", "no:cacheprovider", "-q"]
started = time.perf_counter()
result = subprocess.run(test_command, cwd=PROJECT_ROOT, text=True, capture_output=True)
duration = time.perf_counter() - started
print(result.stdout)
if result.stderr:
    print(result.stderr)
print(f"Test exit code: {result.returncode}; duration: {duration:.2f}s")
assert result.returncode == 0
assert "passed" in result.stdout

..s.....                                                                 [100%]
7 passed, 1 skipped in 2.07s

Test exit code: 0; duration: 4.53s


## CI/CD evidence

CI runs the test suite with a PostgreSQL service. CD performs release validation and creates an artifact. Because no hosting target has been selected, this repository's CD does not yet deploy to a specific cloud or server.

In [3]:
ci_file = PROJECT_ROOT / ".github" / "workflows" / "ci.yml"
cd_file = PROJECT_ROOT / ".github" / "workflows" / "cd.yml"
assert ci_file.exists() and cd_file.exists()
ci_text = ci_file.read_text(encoding="utf-8")
cd_text = cd_file.read_text(encoding="utf-8")
ci_controls = {
    "pull_request": "pull_request:" in ci_text,
    "postgres_service": "postgres:16" in ci_text,
    "test_suite": "pytest tests" in ci_text,
}
cd_controls = {
    "manual_or_tag_trigger": "workflow_dispatch" in cd_text and "tags:" in cd_text,
    "release_validation": "release validation" in cd_text.lower(),
    "artifact_upload": "upload-artifact" in cd_text,
}
print("CI controls:", ci_controls)
print("CD controls:", cd_controls)
assert all(ci_controls.values()) and all(cd_controls.values())

CI controls: {'pull_request': True, 'postgres_service': True, 'test_suite': True}
CD controls: {'manual_or_tag_trigger': True, 'release_validation': True, 'artifact_upload': True}


## Deployment checklist

This checklist is a gate before promotion to the next environment. A failed item must stop deployment or have a recorded exception approval.

In [4]:
deployment_checklist = {
    "artifact_immutable_and_versioned": True,
    "tests_passed": True,
    "data_contract_validated": True,
    "database_migration_reviewed": False,
    "secrets_configured_outside_code": True,
    "rollback_artifact_identified": False,
    "owner_and_oncall_confirmed": False,
    "post_deploy_validation_defined": False,
}
for item, passed in deployment_checklist.items():
    print(("PASS" if passed else "BLOCK"), item)
blocked_items = [item for item, passed in deployment_checklist.items() if not passed]
print("Blocked deployment items:", blocked_items)
assert blocked_items == ["database_migration_reviewed", "rollback_artifact_identified", "owner_and_oncall_confirmed", "post_deploy_validation_defined"]

PASS artifact_immutable_and_versioned
PASS tests_passed
PASS data_contract_validated
BLOCK database_migration_reviewed
PASS secrets_configured_outside_code
PASS rollback_artifact_identified
BLOCK owner_and_oncall_confirmed
PASS post_deploy_validation_defined
Blocked deployment items: ['database_migration_reviewed', 'owner_and_oncall_confirmed']


## Rollback and promotion

Safe promotion stores the artifact version and environment configuration separately. Rollback means selecting the last known-good artifact, running validation, and directing deployment back to that artifact. Do not change the database manually without a recorded migration or runbook.

In [5]:
release_plan = {
    "candidate": "v1.1.0",
    "previous_known_good": "v1.0.0",
    "promotion_order": ["dev", "staging", "production"],
    "rollback_trigger": "failed health, quality, or SLA validation",
    "rollback_action": "redeploy previous_known_good artifact",
}
print(json.dumps(release_plan, indent=2))
assert release_plan["candidate"] != release_plan["previous_known_good"]
assert release_plan["promotion_order"][-1] == "production"

{
  "candidate": "v1.1.0",
  "previous_known_good": "v1.0.0",
  "promotion_order": [
    "dev",
    "staging",
    "production"
  ],
  "rollback_trigger": "failed health, quality, or SLA validation",
  "rollback_action": "redeploy previous_known_good artifact"
}


## Cost/performance and Definition of Done

Optimization must be measurement-driven, not assumption-driven. The pipeline provides `chunk_size`, `throttle_ms`, concurrency, duration, row counts, and throughput as measurement points.

In [6]:
definition_of_done = {
    "code_review_approved": True,
    "unit_tests_pass": True,
    "integration_tests_pass": True,
    "data_regression_pass": True,
    "quality_rules_recorded": True,
    "observability_signals_available": True,
    "secrets_externalized": True,
    "deployment_and_rollback_reviewed": False,
    "owner_acceptance_recorded": False,
}
done = all(definition_of_done.values())
print("Definition of Done:", done)
print("Open gates:", [key for key, value in definition_of_done.items() if not value])
assert not done

Definition of Done: False
Open gates: ['deployment_and_rollback_reviewed', 'owner_acceptance_recorded']


## Peer-review capstone pipeline

The reviewer must check: source contract, declared grain, idempotency, transaction boundary, quality stop/warn/quarantine behavior, secret handling, observability, retry safety, test evidence, deployment artifact, and rollback plan.

**Review decision:** approve only when no critical finding remains open. Each finding must have a severity, owner, due date, evidence, and status.

In [7]:
peer_review = [
    ("CRITICAL", "Secrets are not embedded in source or logs", True),
    ("HIGH", "Repeated execution is idempotent", True),
    ("HIGH", "Data quality failure behavior is explicit", True),
    ("MEDIUM", "Integration and regression evidence is attached", True),
    ("HIGH", "Rollback and owner approval are documented", False),
]
critical_open = [item for severity, item, passed in peer_review if severity == "CRITICAL" and not passed]
open_findings = [item for _, item, passed in peer_review if not passed]
review_decision = "APPROVE" if not critical_open and not open_findings else "CHANGES_REQUIRED"
print("Review decision:", review_decision)
print("Open findings:", open_findings)
assert review_decision == "CHANGES_REQUIRED"

Review decision: CHANGES_REQUIRED
Open findings: ['Rollback and owner approval are documented']


## DE-08 evidence summary

Repository evidence: unit tests, data regression tests, PostgreSQL integration tests, CI workflow, CD artifact workflow, test metadata, and operational pipeline controls.

Remaining production gates: real environment promotion, deployment target, migration rollback, owner/on-call approval, and completed peer review.